# A/B Test: Moving the First Gate (Cookie Cats)

## Context
Cookie Cats is a mobile puzzle game. Progress is periodically blocked by "gates" that make players wait or pay. The company tested moving the first gate from level 30 (control, `gate_30`) to level 40 (treatment, `gate_40`).

## Hypothesis
Moving the gate later (level 40) lets players enjoy the game longer before hitting friction, so it changes player retention.
- H0: the 7-day retention rate is the same in both groups.
- H1: the 7-day retention rate differs between groups (two-sided; I do not assume the direction).

## Metrics (fixed before looking at results)
- **Primary:** `retention_7` (player returned 7 days after install). Retention is the business goal of the gate, and choosing one primary metric up front avoids a multiple-comparisons problem.
- **Secondary:** `retention_1`.
- **Guardrail / context:** `sum_gamerounds` (engagement in the first 14 days). It is heavily skewed, so it needs robust handling.

## Decision rule
- Significance level alpha = 0.05, power = 0.80.
- Ship the change only if the primary metric shows a statistically significant difference AND the confidence interval excludes effects too small to matter.

## Plan
1. Validate the experiment (sample ratio, duplicates, missing values).
2. Power analysis: what effect size can this sample size detect?
3. Test the primary metric, then the secondary metric.
4. Simulate how often sloppy analysis (peeking, many metrics) produces false positives.
5. Recommendation and limitations.

In [1]:
import pandas as pd
from scipy import stats

df = pd.read_csv("../data/cookie_cats.csv")

print(df.shape)
print()
print(df.dtypes)
print()
print("missing values:", df.isna().sum().sum())
print("duplicate userids:", df["userid"].duplicated().sum())
print()

counts = df["version"].value_counts().sort_index()
print(counts)

# Sample ratio mismatch: was the split really 50/50?
chi2, p = stats.chisquare(counts.values)
print(f"\nSample ratio check: chi2 = {chi2:.2f}, p = {p:.4f}")

(90189, 5)

userid            int64
version             str
sum_gamerounds    int64
retention_1        bool
retention_7        bool
dtype: object

missing values: 0
duplicate userids: 0

version
gate_30    44700
gate_40    45489
Name: count, dtype: int64

Sample ratio check: chi2 = 6.90, p = 0.0086


## Experiment validation

- 90,189 users, no missing values, no duplicate user IDs.
- Group sizes: gate_30 = 44,700, gate_40 = 45,489 (50.4% in treatment).
- **Sample ratio mismatch:** chi-square test against a 50/50 split gives p = 0.0086. This fails at the 0.05 level but not at the stricter 0.001 threshold often used for SRM alarms.
- Cause cannot be determined from this dataset (no assignment timestamps or platform data). The imbalance is under 1%, so it is unlikely to drive a large effect, but it is a validity concern that I will carry into the final recommendation.

In [2]:
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p0 = df["retention_7"].mean()
n1, n2 = counts["gate_30"], counts["gate_40"]
ratio = n2 / n1
analysis = NormalIndPower()

# 1) Minimum detectable effect with the sample we actually have
h = analysis.solve_power(effect_size=None, nobs1=n1, alpha=0.05, power=0.80,
                         ratio=ratio, alternative="two-sided")
p1 = np.sin(h / 2 + np.arcsin(np.sqrt(p0))) ** 2   # convert Cohen's h back to a rate
print(f"Baseline 7-day retention (pooled): {p0:.4f}")
print(f"Minimum detectable effect: {100 * (p1 - p0):.2f} percentage points "
      f"({100 * (p1 - p0) / p0:.1f}% relative)")

# 2) Sample size per group needed for different target effects
print("\nUsers needed per group:")
for delta in [0.002, 0.005, 0.01, 0.02]:
    n_needed = analysis.solve_power(effect_size=proportion_effectsize(p0 + delta, p0),
                                    alpha=0.05, power=0.80, ratio=1.0,
                                    alternative="two-sided")
    print(f"  detect +{100 * delta:.1f} pp: {int(np.ceil(n_needed)):,}")

Baseline 7-day retention (pooled): 0.1861
Minimum detectable effect: 0.73 percentage points (3.9% relative)

Users needed per group:
  detect +0.2 pp: 596,791
  detect +0.5 pp: 96,072
  detect +1.0 pp: 24,259
  detect +2.0 pp: 6,183


## Power analysis and decision threshold

- Pooled 7-day retention: 18.6%. With about 45,000 users per group, alpha = 0.05 and power = 0.80, the minimum detectable effect is 0.73 percentage points (3.9% relative).
- Users needed per group: +2.0 pp needs about 6,200; +1.0 pp about 24,300; +0.5 pp about 96,100; +0.2 pp about 596,800.
- The experiment is adequately powered for effects of about 1 pp or more, and underpowered for effects of 0.5 pp or less. A non-significant result would therefore not rule out small effects.
- **Smallest effect worth acting on (assumed, set before seeing results): 0.5 percentage points** of 7-day retention. This is a placeholder and should come from business economics.

In [3]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

def compare(metric):
    c = df.groupby("version")[metric].agg(["sum", "count"])
    x_c, n_c = int(c.loc["gate_30", "sum"]), int(c.loc["gate_30", "count"])
    x_t, n_t = int(c.loc["gate_40", "sum"]), int(c.loc["gate_40", "count"])
    p_c, p_t = x_c / n_c, x_t / n_t

    z, p = proportions_ztest([x_t, x_c], [n_t, n_c])
    lo, hi = confint_proportions_2indep(x_t, n_t, x_c, n_c, compare="diff", method="newcomb")

    print(f"--- {metric} ---")
    print(f"gate_30 (control):   {p_c:.4f}  ({x_c:,} of {n_c:,})")
    print(f"gate_40 (treatment): {p_t:.4f}  ({x_t:,} of {n_t:,})")
    print(f"difference (treatment - control): {100 * (p_t - p_c):+.3f} pp "
          f"(95% CI {100 * lo:+.3f} to {100 * hi:+.3f} pp)")
    print(f"relative change: {100 * (p_t - p_c) / p_c:+.2f}%")
    print(f"z = {z:.3f}, p = {p:.4f}\n")

compare("retention_7")   # primary
compare("retention_1")   # secondary

--- retention_7 ---
gate_30 (control):   0.1902  (8,502 of 44,700)
gate_40 (treatment): 0.1820  (8,279 of 45,489)
difference (treatment - control): -0.820 pp (95% CI -1.328 to -0.312 pp)
relative change: -4.31%
z = -3.164, p = 0.0016

--- retention_1 ---
gate_30 (control):   0.4482  (20,034 of 44,700)
gate_40 (treatment): 0.4423  (20,119 of 45,489)
difference (treatment - control): -0.591 pp (95% CI -1.239 to +0.058 pp)
relative change: -1.32%
z = -1.784, p = 0.0744



## Results: retention

| Metric | gate_30 (control) | gate_40 (treatment) | Difference (95% CI) | p |
|---|---|---|---|---|
| 7-day retention (primary) | 19.02% | 18.20% | -0.82 pp (-1.33 to -0.31) | 0.0016 |
| 1-day retention (secondary) | 44.82% | 44.23% | -0.59 pp (-1.24 to +0.06) | 0.074 |

- Moving the gate from level 30 to level 40 **decreased** 7-day retention by about 0.8 percentage points (4.3% relative). The effect is statistically significant and opposite to the direction the change was meant to improve.
- The confidence interval straddles the pre-set 0.5 pp threshold, so the drop is likely at least 0.3 pp but could be somewhat above or below 0.5 pp.
- 1-day retention shows a similar but non-significant decline.
- **Validity concern:** the unexplained sample ratio mismatch (p = 0.0086) could bias these estimates in an unknown direction.

In [4]:
g = df.groupby("version")["sum_gamerounds"]
print(g.describe(percentiles=[0.5, 0.9, 0.99]).round(1))
print("\nlargest values:", df["sum_gamerounds"].nlargest(5).tolist())
print("share with zero rounds:",
      (df["sum_gamerounds"] == 0).groupby(df["version"]).mean().round(4).to_dict())

a = df.loc[df.version == "gate_30", "sum_gamerounds"].values
b = df.loc[df.version == "gate_40", "sum_gamerounds"].values

# Rank-based test, robust to skew
u, p_mw = stats.mannwhitneyu(b, a, alternative="two-sided")
print(f"\nMann-Whitney U: p = {p_mw:.4f}")

# Welch t-test on log(1 + rounds)
t, p_log = stats.ttest_ind(np.log1p(b), np.log1p(a), equal_var=False)
print(f"Welch t-test on log1p(rounds): p = {p_log:.4f}")

# Bootstrap CI for the difference in means, capping at the 99.9th percentile
cap = np.percentile(df["sum_gamerounds"], 99.9)
a_w, b_w = np.minimum(a, cap), np.minimum(b, cap)
rng = np.random.default_rng(42)
diffs = [rng.choice(b_w, len(b_w)).mean() - rng.choice(a_w, len(a_w)).mean()
         for _ in range(2000)]
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"Capped mean difference (cap = {cap:.0f}): {b_w.mean() - a_w.mean():+.2f} rounds "
      f"(95% bootstrap CI {lo:+.2f} to {hi:+.2f})")

           count  mean    std  min   50%    90%    99%      max
version                                                        
gate_30  44700.0  52.5  256.7  0.0  17.0  135.0  493.0  49854.0
gate_40  45489.0  51.3  103.3  0.0  16.0  134.0  492.1   2640.0

largest values: [49854, 2961, 2640, 2438, 2294]
share with zero rounds: {'gate_30': 0.0433, 'gate_40': 0.0452}

Mann-Whitney U: p = 0.0502
Welch t-test on log1p(rounds): p = 0.0696
Capped mean difference (cap = 1074): -0.10 rounds (95% bootstrap CI -1.29 to +1.20)


## Guardrail: game rounds in the first 14 days

- `sum_gamerounds` is heavily right-skewed. One `gate_30` player logged 49,854 rounds (the next largest value in the data is 2,961), probably a bot or logging error. It inflates the control group's raw standard deviation (256.7 vs 103.3). I kept this user in the data and capped values at the 99.9th percentile (1,074 rounds) for mean comparisons.
- Typical engagement is nearly identical: medians 17 vs 16, 90th percentiles 135 vs 134, zero-round share 4.3% vs 4.5%.
- Mann-Whitney p = 0.050, Welch t-test on log(1 + rounds) p = 0.070, and the capped mean difference is -0.10 rounds (95% bootstrap CI -1.29 to +1.20). I did not specify the engagement test in advance, so all three are reported. They agree: no clear evidence of a change in engagement.
- Engagement does not offset or explain the 7-day retention drop. It is simply flat.

In [6]:
rng = np.random.default_rng(42)
p_true = df["retention_7"].mean()
n_per_group, looks, sims = 45_000, 10, 5_000
step = n_per_group // looks

# --- A/A test: both groups have the SAME retention rate ---
inc_a = rng.binomial(step, p_true, size=(sims, looks)).cumsum(axis=1)
inc_b = rng.binomial(step, p_true, size=(sims, looks)).cumsum(axis=1)
n = step * np.arange(1, looks + 1)

pa, pb = inc_a / n, inc_b / n
pooled = (inc_a + inc_b) / (2 * n)
z = (pb - pa) / np.sqrt(pooled * (1 - pooled) * 2 / n)
pvals = 2 * (1 - stats.norm.cdf(np.abs(z)))

print("A/A test, no true effect, alpha = 0.05")
print(f"  Look once at the planned sample size:  {(pvals[:, -1] < 0.05).mean():.1%} false positives")
print(f"  Peek {looks} times, stop at first p<0.05: {(pvals < 0.05).any(axis=1).mean():.1%} false positives")

# --- Many metrics: 10 independent null metrics, test each at the end ---
metrics = 10
a = rng.binomial(n_per_group, p_true, size=(sims, metrics))
b = rng.binomial(n_per_group, p_true, size=(sims, metrics))
pa, pb = a / n_per_group, b / n_per_group
pooled = (a + b) / (2 * n_per_group)
z = (pb - pa) / np.sqrt(pooled * (1 - pooled) * 2 / n_per_group)
pm = 2 * (1 - stats.norm.cdf(np.abs(z)))

print(f"\n{metrics} metrics tested, no true effect on any:")
print(f"  At least one 'significant' at 0.05:        {(pm < 0.05).any(axis=1).mean():.1%}")
print(f"  With Bonferroni correction (alpha/{metrics}): {(pm < 0.05 / metrics).any(axis=1).mean():.1%}")
print(f"  Theory: 1 - 0.95^{metrics} = {1 - 0.95 ** metrics:.1%}")

A/A test, no true effect, alpha = 0.05
  Look once at the planned sample size:  5.0% false positives
  Peek 10 times, stop at first p<0.05: 19.2% false positives

10 metrics tested, no true effect on any:
  At least one 'significant' at 0.05:        40.3%
  With Bonferroni correction (alpha/10): 4.9%
  Theory: 1 - 0.95^10 = 40.1%


## Why the analysis was planned in advance (false-positive simulation)

Simulated A/A tests (both groups drawn from the same 18.6% retention rate, 45,000 users per group, 5,000 simulations):

| Procedure | False positive rate |
|---|---|
| One planned look at the full sample | 5.0% |
| Peek 10 times, stop at the first p < 0.05 | 19.2% |
| 10 metrics, any one significant at 0.05 | 40.3% (theory: 40.1%) |
| 10 metrics with Bonferroni correction | 4.9% |

Peeking and testing many metrics inflate false positives well beyond the nominal 5%. This is why this analysis fixed one primary metric, the significance level, and the minimum effect of interest before looking at outcomes. With two retention metrics, the Bonferroni threshold is 0.025, and the primary result (p = 0.0016) clears it.